# IDPFold2 Monomer Preview (Colab)

This notebook runs IDPFold2 for the bundled example CSV or FASTA text, and previews the generated `.pdb` file in 3D.

Use **Runtime > Change runtime type > GPU** for practical inference speed.

In [ ]:
#@title <b>Preliminary operations</b>: setting the environment (i)
# Run this cell by itself. condacolab restarts the runtime when installation finishes.
import subprocess
subprocess.run( 'pip install -q condacolab'.split())

import condacolab
condacolab.install()

In [ ]:
#@title <b>Preliminary operations</b>: setting the environment (ii)
# Run this cell after the condacolab runtime restart has completed.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Junjie-Zhu/IDPFold2'
REPO_DIR = Path('/content/IDPFold-multimer')


def run(command):
    print('$', ' '.join(str(part) for part in command))
    subprocess.run(command, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])

PIP_DEPS = [
    'torch==2.4.1',
    'einops==0.6',
    'dm-tree==0.1.8',
    'loguru==0.7.2',
    'hydra-core==1.3.1',
    'pandas',
    'numpy==1.26.0',
    'biotite==0.41.0',
    'biopandas==0.5.1',
    'wget==3.2',
    'tqdm==4.66.4',
    'cpdb-protein',
    'biopython',
    'rootutils',
    'pytest',
]
run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *PIP_DEPS])

# Runtime/UI extras needed by this Colab notebook but not listed in environment.yaml.
NOTEBOOK_DEPS = ['fair-esm', 'scipy', 'py3Dmol', 'ipywidgets']
run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *NOTEBOOK_DEPS])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'])

if not (REPO_DIR / 'src' / 'inference.py').exists():
    raise FileNotFoundError(f'IDPFold2 source tree was not found under {REPO_DIR}')

os.chdir(REPO_DIR)
repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import numpy as np
import pandas as pd
import scipy
import torch
import esm
import hydra
import py3Dmol
import src.inference

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Environment is ready:', REPO_DIR)

In [ ]:
#@title Download the IDPFold2 inference checkpoint
import urllib.request

CHECKPOINT_URL = 'https://zenodo.org/records/18239596/files/IDPFold2_ema_0.999_260114.pth?download=1'
CHECKPOINT_DIR = Path('/content/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = CHECKPOINT_DIR / 'IDPFold2_ema_0.999_260114.pth'

if not CKPT_PATH.exists():
    print('Downloading checkpoint...')
    urllib.request.urlretrieve(CHECKPOINT_URL, CKPT_PATH)

if not CKPT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint download failed: {CKPT_PATH}')

print('Checkpoint:', CKPT_PATH)

In [ ]:
import re
import pandas as pd
from google.colab import files

WORK_DIR = REPO_DIR
INPUT_CSV = Path('/content/input_monomer.csv')
ALLOWED_AA = set('ACDEFGHIKLMNPQRSTVWYX')

INPUT_MODE = 'example'  # @param ['example', 'fasta_text', 'custom_sequence', 'csv_upload']

# used when INPUT_MODE='example'; use 'all' to run every row in data/monomer_example.csv
EXAMPLE_TEST_CASE = 'THB_C2'  # @param {type:'string'}

# used when INPUT_MODE='fasta_text'
FASTA_TEXT = '>custom_demo\nGPGSEDVWEILRQAPPSEYERIAFQYGVTDLRGMLKRLKGMRRDEKKSTAFQKKLEPAYQVSKGHKIRLTVELADHDAEVKWLKNGQEIQMSGSKYIFESIGAKRTLTISQCSLADDAAYQCVVGGEKCSTELFVKE'  # @param {type:'string'}

# used when INPUT_MODE='custom_sequence'
CUSTOM_TEST_CASE = 'custom_demo'  # @param {type:'string'}
CUSTOM_SEQUENCE = 'GPGSEDVWEILRQAPPSEYERIAFQYGVTDLRGMLKRLKGMRRDEKKSTAFQKKLEPAYQVSKGHKIRLTVELADHDAEVKWLKNGQEIQMSGSKYIFESIGAKRTLTISQCSLADDAAYQCVVGGEKCSTELFVKE'  # @param {type:'string'}


def sanitize_name(name):
    name = re.sub(r'[^A-Za-z0-9_.-]+', '_', name.strip())
    return name or 'custom_demo'


def normalize_sequence(sequence):
    return re.sub(r'\s+', '', sequence).upper()


def parse_fasta(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        raise ValueError('FASTA input is empty.')
    header = 'custom_demo'
    seq_lines = []
    for line in lines:
        if line.startswith('>'):
            if seq_lines:
                break
            header = line[1:].split()[0] or header
        else:
            seq_lines.append(line)
    return sanitize_name(header), normalize_sequence(''.join(seq_lines))


def validate_inputs(df):
    required = {'test_case', 'sequence'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Input CSV is missing columns: {sorted(missing)}')
    df = df.loc[:, ['test_case', 'sequence']].copy()
    df['test_case'] = df['test_case'].map(lambda value: sanitize_name(str(value)))
    df['sequence'] = df['sequence'].map(lambda value: normalize_sequence(str(value)))
    if df['test_case'].duplicated().any():
        raise ValueError('test_case values must be unique.')
    for row in df.itertuples(index=False):
        if not row.sequence:
            raise ValueError(f'{row.test_case} has an empty sequence.')
        invalid = sorted(set(row.sequence) - ALLOWED_AA)
        if invalid:
            raise ValueError(f'{row.test_case} contains unsupported residue codes: {invalid}')
    return df


if INPUT_MODE == 'example':
    example_csv = WORK_DIR / 'data' / 'monomer_example.csv'
    input_df = pd.read_csv(example_csv)
    if EXAMPLE_TEST_CASE.strip().lower() != 'all':
        input_df = input_df[input_df['test_case'] == EXAMPLE_TEST_CASE.strip()]
        if input_df.empty:
            raise ValueError(f'Example test case not found: {EXAMPLE_TEST_CASE}')
elif INPUT_MODE == 'fasta_text':
    test_case, sequence = parse_fasta(FASTA_TEXT)
    input_df = pd.DataFrame([{'test_case': test_case, 'sequence': sequence}])
elif INPUT_MODE == 'custom_sequence':
    input_df = pd.DataFrame([{'test_case': CUSTOM_TEST_CASE, 'sequence': CUSTOM_SEQUENCE}])
elif INPUT_MODE == 'csv_upload':
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('No CSV file uploaded.')
    uploaded_name = next(iter(uploaded))
    input_df = pd.read_csv(uploaded_name)
else:
    raise ValueError(f'Unsupported INPUT_MODE: {INPUT_MODE}')

input_df = validate_inputs(input_df)
input_df.to_csv(INPUT_CSV, index=False)
EXPECTED_TEST_CASES = input_df['test_case'].tolist()

print('Input CSV:', INPUT_CSV)
print('Test cases:', ', '.join(EXPECTED_TEST_CASES))
display(input_df)

In [ ]:
import shlex
import shutil

PREFIX = 'COLAB_MONOMER'  # @param {type:'string'}
NSAMPLES = 4              # @param {type:'integer'}
MAX_BATCH_LENGTH = 3500   # @param {type:'integer'}
NUM_WORKERS = 0           # @param {type:'integer'}

PLM_EMB_DIR = Path('/content/embeddings')
LOGGING_DIR = Path('/content/outputs')
PLM_EMB_DIR.mkdir(parents=True, exist_ok=True)
LOGGING_DIR.mkdir(parents=True, exist_ok=True)

entrypoint = ['idpfold2-infer'] if shutil.which('idpfold2-infer') else [sys.executable, str(WORK_DIR / 'src' / 'inference.py')]
cmd = entrypoint + [
    f'prefix={PREFIX}',
    f'ckpt_dir={CKPT_PATH}',
    f'plm_emb_dir={PLM_EMB_DIR}',
    f'csv_dir={INPUT_CSV}',
    f'nsamples={NSAMPLES}',
    f'max_batch_length={MAX_BATCH_LENGTH}',
    f'num_workers={NUM_WORKERS}',
    f'logging_dir={LOGGING_DIR}',
]

print('Running:', ' '.join(shlex.quote(str(x)) for x in cmd))
subprocess.run(cmd, check=True, cwd=WORK_DIR)

run_dirs = sorted(
    [path for path in LOGGING_DIR.glob(f'{PREFIX}_INF_*') if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
)
if not run_dirs:
    raise FileNotFoundError(f'No output directory was created under {LOGGING_DIR}')

LATEST_RUN_DIR = run_dirs[-1]
PDB_FILES = {path.stem: path for path in sorted((LATEST_RUN_DIR / 'samples').glob('*.pdb'))}
if not PDB_FILES:
    raise FileNotFoundError(f'No PDB files were generated in {LATEST_RUN_DIR / "samples"}')

print('Output directory:', LATEST_RUN_DIR)
print('Generated PDB files:')
for name, path in PDB_FILES.items():
    print(f'  {name}: {path}')

In [ ]:
import zipfile
import py3Dmol
from google.colab import files

# Leave empty to view the first generated test case.
VIEW_TEST_CASE = ''  # @param {type:'string'}
DOWNLOAD_ALL = False  # @param {type:'boolean'}

selected_name = VIEW_TEST_CASE.strip() or next(iter(PDB_FILES))
if selected_name not in PDB_FILES:
    raise ValueError(f'Unknown VIEW_TEST_CASE {selected_name!r}. Available: {list(PDB_FILES)}')

PDB_PATH = PDB_FILES[selected_name]
pdb_text = PDB_PATH.read_text()
viewer = py3Dmol.view(width=900, height=600)
viewer.addModel(pdb_text, 'pdb')
viewer.setStyle({'cartoon': {'color': 'spectrum'}})
viewer.zoomTo()
viewer.show()

print(f'Selected PDB: {PDB_PATH}')
if DOWNLOAD_ALL and len(PDB_FILES) > 1:
    ZIP_PATH = LATEST_RUN_DIR / 'idpfold2_colab_pdbs.zip'
    with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in PDB_FILES.values():
            archive.write(path, arcname=path.name)
    print(f'Download all PDB files: {ZIP_PATH}')
    files.download(str(ZIP_PATH))
else:
    print(f'Download selected PDB: {PDB_PATH}')
    files.download(str(PDB_PATH))